# Activity 4 — Debug tool calling

This is Activity 4 of the LLM Observability Hackathon. Run the cells top-to-bottom in a fresh kernel. The broken tool is intentional; do not fix it before inspecting its trace.

## Purpose
Diagnose a failing tool call from trace data and repair the tool implementation. The trace records the tool arguments, the tool span, and the tool result or error.

## What to expect
The broken tool reads `booking_id`, but the schema sends `booking_ref`, so the tool span is `ERROR` with `KeyError: 'booking_id'` and the answer reports the failure. The fixed tool reads `booking_ref` and returns the booking. The notebook creates the traces `activity-4-tool-call-broken` and `activity-4-tool-call-fixed`.

## Bad practice
Guess the tool argument name and ignore the tool span. The model and the implementation disagree, and the failure is only visible in the final answer.

## Good practice
Align the tool schema with the implementation, then inspect the tool span's arguments, level, and error when a call fails.

## Why the good practice works
The tool span shows exactly which argument the model sent and which argument the implementation read. The mismatch and the exception are recorded in the trace, so the fix targets the implementation rather than the prompt.

## Deliverable
Both trace URLs, the tool-span evidence, and the implementation change.


In [1]:
import json
from IPython.display import Markdown, display
from experiment_utils import (CORPUS, client, langfuse, MODEL, RUN_ID, print_summary,
                              propagate_attributes, record_trace, show_trace)


In [2]:
BOOKINGS = CORPUS["bookings"]
TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_booking",
        "description": "Look up a booking by reference",
        "parameters": {
            "type": "object",
            "properties": {"booking_ref": {"type": "string"}},
            "required": ["booking_ref"],
        },
    },
}

def get_booking(**kwargs):
    ref = kwargs["booking_id"]
    return BOOKINGS[ref]

def run_tool_loop(question, variant):
    trace_name = f"activity-4-tool-call-{variant}"
    tool_args = None
    tool_result = None
    answer = None
    with langfuse.start_as_current_observation(name=trace_name, as_type="span") as root:
        with propagate_attributes(
            session_id=f"activity-4-{RUN_ID}",
            tags=["activity-4", variant],
            trace_name=trace_name,
        ):
            messages = [
                {"role": "system", "content": "You are Skyline Travel's assistant. Use the get_booking tool to look up bookings. If a tool call fails, report the failure to the user instead of retrying."},
                {"role": "user", "content": question},
            ]
            first = client.chat.completions.create(
                model=MODEL, messages=messages, tools=[TOOL_SCHEMA], tool_choice="auto",
                temperature=0, max_tokens=800, name="activity-4-generation-1",
            )
            message = first.choices[0].message
            if message.tool_calls:
                call = message.tool_calls[0]
                tool_args = json.loads(call.function.arguments)
                messages.append(message.model_dump(exclude_none=True))
                with langfuse.start_as_current_observation(
                    name="get_booking", as_type="tool", input=tool_args
                ) as tool_span:
                    try:
                        tool_result = get_booking(**tool_args)
                        tool_span.update(output=tool_result)
                    except Exception as exc:
                        tool_result = {"error": f"{type(exc).__name__}: {exc}"}
                        tool_span.update(
                            level="ERROR",
                            status_message=f"{type(exc).__name__}: {exc}",
                            output=tool_result,
                        )
                messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(tool_result)})
                second = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=[TOOL_SCHEMA], tool_choice="auto",
                    temperature=0, max_tokens=800, name="activity-4-generation-2",
                )
                answer = second.choices[0].message.content
            else:
                answer = message.content
            trace_id = langfuse.get_current_trace_id()
            root.update(output={"answer": answer, "tool_args": tool_args, "tool_result": tool_result})
    return {
        "variant": variant,
        "answer": answer,
        "tool_args": tool_args,
        "tool_result": tool_result,
        "trace_id": trace_id,
        "trace_url": record_trace(f"activity-4-{variant}", trace_id),
    }

question = "What is the status of booking SKY-1042?"
broken = run_tool_loop(question, "broken")
print("tool arguments:", broken["tool_args"])
print("tool result:", broken["tool_result"])
print("final answer:", broken["answer"])
show_trace(broken["trace_id"])

tool arguments: {'booking_ref': 'SKY-1042'}
tool result: {'error': "KeyError: 'booking_id'"}
final answer: I tried to look up booking **SKY-1042**, but the lookup failed with this error:

> `KeyError: 'booking_id'`

This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more.
trace: http://localhost:3000/project/langfuse-observability/traces/9875b1e644eb8c0349fc48997b9e2730


'http://localhost:3000/project/langfuse-observability/traces/9875b1e644eb8c0349fc48997b9e2730'

## What went wrong?

Open the broken trace and inspect the `get_booking` tool span. Compare the arguments the model sent with the arguments the implementation tried to read. Fix the implementation, not the prompt.


In [3]:
def get_booking(**kwargs):
    ref = kwargs["booking_ref"]
    return BOOKINGS[ref]

fixed = run_tool_loop(question, "fixed")
print("tool arguments:", fixed["tool_args"])
print("tool result:", fixed["tool_result"])
print("final answer:", fixed["answer"])
show_trace(fixed["trace_id"])

rows = ["| variant | tool arguments | tool result/error | final answer | trace |",
        "|---|---|---|---|---|"]
for run in [broken, fixed]:
    args = json.dumps(run["tool_args"]).replace("|", "\\|")
    result = json.dumps(run["tool_result"]).replace("|", "\\|")
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['variant']} | {args} | {result} | {answer} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))

tool arguments: {'booking_ref': 'SKY-1042'}
tool result: {'status': 'confirmed', 'origin': 'Lisbon', 'destination': 'Madrid', 'date': '2026-12-01', 'passengers': 2, 'cabin': 'economy'}
final answer: Booking **SKY-1042** is **confirmed**.

- **Route:** Lisbon → Madrid
- **Date:** December 1, 2026
- **Passengers:** 2
- **Cabin:** Economy

Let me know if you'd like any other details about this booking.
trace: http://localhost:3000/project/langfuse-observability/traces/3b6a65b802b959bc35b00dd7a3697ca9


| variant | tool arguments | tool result/error | final answer | trace |
|---|---|---|---|---|
| broken | {"booking_ref": "SKY-1042"} | {"error": "KeyError: 'booking_id'"} | I tried to look up booking **SKY-1042**, but the lookup failed with this error:  > `KeyError: 'booking_id'`  This appears to be a problem on our end rather than an issue with your booking reference. I haven't retried the lookup — please try again shortly, or let me know if you'd like me to attempt it once more. | http://localhost:3000/project/langfuse-observability/traces/9875b1e644eb8c0349fc48997b9e2730 |
| fixed | {"booking_ref": "SKY-1042"} | {"status": "confirmed", "origin": "Lisbon", "destination": "Madrid", "date": "2026-12-01", "passengers": 2, "cabin": "economy"} | Booking **SKY-1042** is **confirmed**.  - **Route:** Lisbon → Madrid - **Date:** December 1, 2026 - **Passengers:** 2 - **Cabin:** Economy  Let me know if you'd like any other details about this booking. | http://localhost:3000/project/langfuse-observability/traces/3b6a65b802b959bc35b00dd7a3697ca9 |

## Wrap-up

Run this cell to print the session id and trace URLs for this activity, then flush the Langfuse client. Submit the trace URLs and your findings.


In [4]:
print_summary(4)
langfuse.flush()
print("flushed")

run id: 8f597fb4
session activity-4-8f597fb4
activity-4-broken: http://localhost:3000/project/langfuse-observability/traces/9875b1e644eb8c0349fc48997b9e2730
activity-4-fixed: http://localhost:3000/project/langfuse-observability/traces/3b6a65b802b959bc35b00dd7a3697ca9
flushed
